In [1]:
def jaccard_similarity(a: int, b: int) -> float:
    u = a | b
    return (a & b).bit_count() / u.bit_count() if u else 1.0


def name(combo):
    return "{" + ", ".join(f"a{i+1}" for i in sorted(idx[m] for m in combo)) + "}"

In [2]:
import random
import math
import statistics


def generate_subsets(
    n: int = 75,
    mean_len: float = 10.0,
    std_len: float = 3.0,
    *,
    universe_start: int = 1,
    universe_size: int = 40,
    dirty_prob: float = 0.2,
    outside_pool: list[int] | None = None,
    seed: int | None = None,
    max_attempts: int = 500,
):
    """
    Генерирует n подмножеств с длинами ~ N(mean_len, std_len^2).
    Границы длины не задаются: они выводятся из универсума —
    минимум 1 (или 2, если элемент может быть «грязным»),
    максимум = universe_size (+1, если допускается «грязный» элемент).

    Параметры
    ---------
    n            : число подмножеств
    mean_len     : средняя длина
    std_len      : стандартное отклонение длины (СКО)
    universe_start, universe_size : универсум = {universe_start .. universe_start+universe_size-1}
    dirty_prob   : вероятность, что подмножество содержит элемент вне U
    outside_pool : откуда брать «грязные» элементы; по умолчанию
                   range(universe_start+universe_size, ...+10)
    seed         : для воспроизводимости
    max_attempts : попыток на уникальность одного подмножества

    Возвращает
    ----------
    universe : set[int]
    subsets  : list[frozenset[int]]
    idx      : dict[frozenset[int], int]
    """
    if outside_pool is None:
        outside_pool = list(range(
            universe_start + universe_size,
            universe_start + universe_size + 10,
        ))
    outside_pool = list(outside_pool)

    if std_len < 0:
        raise ValueError("std_len должен быть >= 0")
    if not (0.0 <= dirty_prob <= 1.0):
        raise ValueError("dirty_prob должен быть в [0, 1]")

    # --- границы длины выводим из универсума ---
    min_len = 2 if dirty_prob > 0 else 1
    max_len = universe_size + (1 if dirty_prob > 0 else 0)

    rng = random.Random(seed)
    universe = set(range(universe_start, universe_start + universe_size))
    clean_pool = sorted(universe)

    def sample_len() -> int:
        if std_len == 0:
            return max(min_len, min(max_len, int(round(mean_len))))
        for _ in range(200):
            L = int(round(rng.gauss(mean_len, std_len)))
            if min_len <= L <= max_len:
                return L
        return max(min_len, min(max_len, int(round(mean_len))))

    subsets: list[frozenset[int]] = []
    seen: set[frozenset[int]] = set()

    for _ in range(n):
        L = sample_len()
        is_dirty = rng.random() < dirty_prob

        placed = False
        for _ in range(max_attempts):
            if is_dirty:
                L_eff = max(L, 2)
                outside_el = rng.choice(outside_pool)
                inside = rng.sample(clean_pool, min(L_eff - 1, universe_size))
                s = frozenset(inside + [outside_el])
            else:
                s = frozenset(rng.sample(clean_pool, min(L, universe_size)))
            if s not in seen:
                seen.add(s)
                subsets.append(s)
                placed = True
                break
        if not placed:
            subsets.append(s)
            seen.add(s)

    idx = {s: i + 1 for i, s in enumerate(subsets)}
    return universe, subsets, idx


def report_lengths(subsets):
    lens = [len(s) for s in subsets]
    print(f"n          = {len(lens)}")
    print(f"mean_len   = {statistics.mean(lens):.3f}")
    print(f"std_len    = {statistics.pstdev(lens):.3f}")
    print(f"min / max  = {min(lens)} / {max(lens)}")

In [3]:
universe, subsets, idx = generate_subsets(
    n=30,
    mean_len=4,        # близко к min_len=2 → мода слева
    std_len=2,         # большой разброс → правый хвост дотянется до max_len=31
    universe_size=30,
    dirty_prob=0,
    seed=42,
)
report_lengths(subsets)

n          = 30
mean_len   = 3.867
std_len    = 1.310
min / max  = 1 / 7


In [4]:
# --- 1. Перевод универсума и подмножеств в битовые маски ---
def to_mask(s, base=0):
    m = 0
    for x in s:
        m |= 1 << (x - base)
    return m

base = min(universe)                 # обычно 1
U_mask = to_mask(universe, base)
S_masks = [to_mask(s, base) for s in subsets]   # порядок совпадает с subsets

# --- 2. Jaccard через popcount ---
def jaccard_masks(a: int, b: int) -> float:
    u = a | b
    return (a & b).bit_count() / u.bit_count() if u else 1.0

# --- 3. Метрика первого слоя ---
metrics = {frozenset([S_masks[i]]): jaccard_masks(U_mask, S_masks[i])
           for i in range(len(S_masks))}

In [5]:
# вместо idx по frozenset — idx по маске
idx = {S_masks[i]: i for i in range(len(S_masks))}

best_score = max(metrics.values())
best_comb = [c for c, v in metrics.items() if v == best_score]
show_matrix = False
level = 1

while metrics and best_score < 1.0:
    new_metrics = {}

    if show_matrix:
        print(f"\n# {level}")
        print(" " * 16 + "   ".join(f"a{i + 1}" for i in range(len(S_masks))))

    for combo, score in metrics.items():
        covered = 0
        for m in combo:
            covered |= m
        max_i = max(idx[m] for m in combo)
        row = []

        for j in range(len(S_masks)):
            s = S_masks[j]
            if j <= max_i:
                row.append(" ")
                continue
            if s & ~covered == 0:          # s ⊆ covered  (эквивалент s <= covered)
                row.append("-")
                continue

            new_score = jaccard_masks(U_mask, covered | s)
            if new_score > score:
                row.append("+")
                new_metrics[combo | {s}] = new_score
            else:
                row.append("-")

        if show_matrix:
            print(f"{name(combo):<16}" + "    ".join(row))

    if new_metrics:
        layer_best = max(new_metrics.values())
        if layer_best > best_score:
            best_score = layer_best
            best_comb = [c for c, v in new_metrics.items() if v == layer_best]

    print(f"слой {level}: комбинаций {len(metrics)}, лучшая схожесть {best_score:.3f}")
    metrics = new_metrics
    level += 1

print(f"\nЛучшее покрытие, схожесть = {best_score:.3f}")
for c in best_comb:
    print(name(c))

слой 1: комбинаций 30, лучшая схожесть 0.400
слой 2: комбинаций 428, лучшая схожесть 0.533
слой 3: комбинаций 3807, лучшая схожесть 0.633
слой 4: комбинаций 23339, лучшая схожесть 0.733
слой 5: комбинаций 103579, лучшая схожесть 0.833
слой 6: комбинаций 341618, лучшая схожесть 0.900
слой 7: комбинаций 847554, лучшая схожесть 0.933
слой 8: комбинаций 1583603, лучшая схожесть 0.967
слой 9: комбинаций 2208355, лучшая схожесть 0.967
слой 10: комбинаций 2253714, лучшая схожесть 0.967
слой 11: комбинаций 1629416, лучшая схожесть 0.967
слой 12: комбинаций 793500, лучшая схожесть 0.967
слой 13: комбинаций 240078, лучшая схожесть 0.967
слой 14: комбинаций 39162, лучшая схожесть 0.967
слой 15: комбинаций 2578, лучшая схожесть 0.967
слой 16: комбинаций 37, лучшая схожесть 0.967

Лучшее покрытие, схожесть = 0.967
{a1, a4, a6, a7, a8, a9, a24, a25, a28}
{a1, a4, a6, a7, a9, a16, a24, a25, a28}
{a1, a6, a7, a8, a9, a12, a24, a25, a28}
{a1, a6, a7, a8, a9, a14, a24, a25, a28}
{a1, a6, a7, a8, a9, a17